# Visualize HDF5 in a knowledge graph

Use [PyVis](https://pyvis.readthedocs.io/) to explore the RDF graph. Drag nodes and zoom to navigate. The saved interactive outputs are included in the documentation; rerunning the notebook requires PyVis.

In [ ]:
# %pip install pyvis

In [ ]:
from html import escape
import re
from IPython.display import display
from pyvis.network import Network
import rdflib
import numpy as np

In [ ]:
import h5rdmtoolbox as h5tbx

In [ ]:
with h5tbx.File() as h5:
    h5.create_dataset(name='test', data=4.3)
    h5.create_dataset(name='nd-array', data=np.random.rand(2, 3), chunks=(1,3))
    grp = h5.create_group(name='contact')
    grp.attrs['fname', rdflib.FOAF.firstName] = 'Matthias'
    grp.attrs['lname', rdflib.FOAF.lastName] = 'Probst'
    grp.attrs['hint'] = 'This group could be representing a person.'
    grp.rdf['hint'].definition = 'A hint gives helpful information on something.'
    grp.rdf.type = rdflib.PROV.Person  # --> rdf.type = Person
    grp.rdf.subject = 'https://orcid.org/0000-0001-8729-0482'  # -> @id='https://orcid.org/0000-0001-8729-0482'
    
graph = h5tbx.get_ld(h5.hdf_filename, structural=True, file_uri="https://example.org#", context={"ex": "https://example.org#"})
print(graph.serialize())

In [ ]:
def show_graph(graph):
    network = Network(height="600px", width="100%", directed=True, cdn_resources="in_line")
    for subject, predicate, obj in graph:
        for term in (subject, obj):
            label = graph.namespace_manager.normalizeUri(term) if isinstance(term, rdflib.URIRef) else str(term)
            network.add_node(term.n3(), label=label)
        network.add_edge(subject.n3(), obj.n3(), label=graph.namespace_manager.normalizeUri(predicate))
    network.force_atlas_2based(overlap=0)
    html = network.generate_html()
    html = re.sub(r'<link\s+href="https://cdn\.jsdelivr\.net/npm/bootstrap@.*?/>|<script\s+src="https://cdn\.jsdelivr\.net/npm/bootstrap@.*?</script>', '', html, flags=re.DOTALL)
    display({'text/html': f'<iframe title="Interactive RDF graph" sandbox="allow-scripts" style="width:100%;height:620px;border:0" srcdoc="{escape(html, quote=True)}"></iframe>'}, raw=True)

In [ ]:
show_graph(graph)

<iframe title="Interactive RDF graph" sandbox="allow-scripts" style="width:100%;height:620px;border:0" srcdoc="<html>
 <head>
 <meta charset="utf-8">
 
 <script>function neighbourhoodHighlight(params) {
 // console.log("in nieghbourhoodhighlight");
 allNodes = nodes.get({ returnType: "Object" });
 // originalNodes = JSON.parse(JSON.stringify(allNodes));
 // if something is selected:
 if (params.nodes.length > 0) {
 highlightActive = true;
 var i, j;
 var selectedNode = params.nodes[0];
 var degrees = 2;

 // mark all nodes as hard to read.
 for (let nodeId in allNodes) {
 // nodeColors[nodeId] = allNodes[nodeId].color;
 allNodes[nodeId].color = "rgba(200,200,200,0.5)";
 if (allNodes[nodeId].hiddenLabel === undefined) {
 allNodes[nodeId].hiddenLabel = allNodes[nodeId].label;
 allNodes[nodeId].label = undefined;
 }
 }
 var connectedNodes = network.getConnectedNodes(selectedNode);
 var allConnectedNodes = [];

 // get the second degree nodes
 for (i = 1; i < degrees; i++) {
 for (j = 0; j < connectedNodes.length; j++) {
 allConnectedNodes = allConnectedNodes.concat(
 network.getConnectedNodes(connectedNodes[j])
 );
 }
 }

 // all second degree nodes get a different color and their label back
 for (i = 0; i < allConnectedNodes.length; i++) {
 // allNodes[allConnectedNodes[i]].color = "pink";
 allNodes[allConnectedNodes[i]].color = "rgba(150,150,150,0.75)";
 if (allNodes[allConnectedNodes[i]].hiddenLabel !== undefined) {
 allNodes[allConnectedNodes[i]].label =
 allNodes[allConnectedNodes[i]].hiddenLabel;
 allNodes[allConnectedNodes[i]].hiddenLabel = undefined;
 }
 }

 // all first degree nodes get their own color and their label back
 for (i = 0; i < connectedNodes.length; i++) {
 // allNodes[connectedNodes[i]].color = undefined;
 allNodes[connectedNodes[i]].color = nodeColors[connectedNodes[i]];
 if (allNodes[connectedNodes[i]].hiddenLabel !== undefined) {
 allNodes[connectedNodes[i]].label =
 allNodes[connectedNodes[i]].hiddenLabel;
 allNodes[connectedNodes[i]].hiddenLabel = undefined;
 }
 }

 // the main node gets its own color and its label back.
 // allNodes[selectedNode].color = undefined;
 allNodes[selectedNode].color = nodeColors[selectedNode];
 if (allNodes[selectedNode].hiddenLabel !== undefined) {
 allNodes[selectedNode].label = allNodes[selectedNode].hiddenLabel;
 allNodes[selectedNode].hiddenLabel = undefined;
 }
 } else if (highlightActive === true) {
 // console.log("highlightActive was true");
 // reset all nodes
 for (let nodeId in allNodes) {
 // allNodes[nodeId].color = "purple";
 allNodes[nodeId].color = nodeColors[nodeId];
 // delete allNodes[nodeId].color;
 if (allNodes[nodeId].hiddenLabel !== undefined) {
 allNodes[nodeId].label = allNodes[nodeId].hiddenLabel;
 allNodes[nodeId].hiddenLabel = undefined;
 }
 }
 highlightActive = false;
 }

 // transform the object into an array
 var updateArray = [];
 if (params.nodes.length > 0) {
 for (let nodeId in allNodes) {
 if (allNodes.hasOwnProperty(nodeId)) {
 // console.log(allNodes[nodeId]);
 updateArray.push(allNodes[nodeId]);
 }
 }
 nodes.update(updateArray);
 } else {
 // console.log("Nothing was selected");
 for (let nodeId in allNodes) {
 if (allNodes.hasOwnProperty(nodeId)) {
 // console.log(allNodes[nodeId]);
 // allNodes[nodeId].color = {};
 updateArray.push(allNodes[nodeId]);
 }
 }
 nodes.update(updateArray);
 }
}

function filterHighlight(params) {
 allNodes = nodes.get({ returnType: "Object" });
 // if something is selected:
 if (params.nodes.length > 0) {
 filterActive = true;
 let selectedNodes = params.nodes;

 // hiding all nodes and saving the label
 for (let nodeId in allNodes) {
 allNodes[nodeId].hidden = true;
 if (allNodes[nodeId].savedLabel === undefined) {
 allNodes[nodeId].savedLabel = allNodes[nodeId].label;
 allNodes[nodeId].label = undefined;
 }
 }

 for (let i=0; i < selectedNodes.length; i++) {
 allNodes[selectedNodes[i]].hidden = false;
 if (allNodes[selectedNodes[i]].savedLabel !== undefined) {
 allNodes[selectedNodes[i]].label = allNod

The first graph includes the HDF5 structure. The next graph contains only the contextual metadata.

In [ ]:
with h5tbx.File() as h5:
    h5.create_dataset(name='test', data=4.3)
    grp = h5.create_group(name='contact')
    grp.attrs['fname', rdflib.FOAF.firstName] = 'Matthias'
    grp.attrs['lname', rdflib.FOAF.lastName] = 'Probst'
    grp.attrs['hint'] = 'bah blah'
    grp.rdf['hint'].definition = 'A hint'
    grp.rdf.subject = rdflib.PROV.Person
    grp.attrs['@id'] = 'https://orcid.org/0000-0001-8729-0482'
    print(h5.dump_jsonld(indent=2, structural=False, file_uri="https://example.org#"))

graph = h5tbx.get_ld(h5.hdf_filename, structural=False, file_uri="https://example.org#")

In [ ]:
show_graph(graph)

<iframe title="Interactive RDF graph" sandbox="allow-scripts" style="width:100%;height:620px;border:0" srcdoc="<html>
 <head>
 <meta charset="utf-8">
 
 <script>function neighbourhoodHighlight(params) {
 // console.log("in nieghbourhoodhighlight");
 allNodes = nodes.get({ returnType: "Object" });
 // originalNodes = JSON.parse(JSON.stringify(allNodes));
 // if something is selected:
 if (params.nodes.length > 0) {
 highlightActive = true;
 var i, j;
 var selectedNode = params.nodes[0];
 var degrees = 2;

 // mark all nodes as hard to read.
 for (let nodeId in allNodes) {
 // nodeColors[nodeId] = allNodes[nodeId].color;
 allNodes[nodeId].color = "rgba(200,200,200,0.5)";
 if (allNodes[nodeId].hiddenLabel === undefined) {
 allNodes[nodeId].hiddenLabel = allNodes[nodeId].label;
 allNodes[nodeId].label = undefined;
 }
 }
 var connectedNodes = network.getConnectedNodes(selectedNode);
 var allConnectedNodes = [];

 // get the second degree nodes
 for (i = 1; i < degrees; i++) {
 for (j = 0; j < connectedNodes.length; j++) {
 allConnectedNodes = allConnectedNodes.concat(
 network.getConnectedNodes(connectedNodes[j])
 );
 }
 }

 // all second degree nodes get a different color and their label back
 for (i = 0; i < allConnectedNodes.length; i++) {
 // allNodes[allConnectedNodes[i]].color = "pink";
 allNodes[allConnectedNodes[i]].color = "rgba(150,150,150,0.75)";
 if (allNodes[allConnectedNodes[i]].hiddenLabel !== undefined) {
 allNodes[allConnectedNodes[i]].label =
 allNodes[allConnectedNodes[i]].hiddenLabel;
 allNodes[allConnectedNodes[i]].hiddenLabel = undefined;
 }
 }

 // all first degree nodes get their own color and their label back
 for (i = 0; i < connectedNodes.length; i++) {
 // allNodes[connectedNodes[i]].color = undefined;
 allNodes[connectedNodes[i]].color = nodeColors[connectedNodes[i]];
 if (allNodes[connectedNodes[i]].hiddenLabel !== undefined) {
 allNodes[connectedNodes[i]].label =
 allNodes[connectedNodes[i]].hiddenLabel;
 allNodes[connectedNodes[i]].hiddenLabel = undefined;
 }
 }

 // the main node gets its own color and its label back.
 // allNodes[selectedNode].color = undefined;
 allNodes[selectedNode].color = nodeColors[selectedNode];
 if (allNodes[selectedNode].hiddenLabel !== undefined) {
 allNodes[selectedNode].label = allNodes[selectedNode].hiddenLabel;
 allNodes[selectedNode].hiddenLabel = undefined;
 }
 } else if (highlightActive === true) {
 // console.log("highlightActive was true");
 // reset all nodes
 for (let nodeId in allNodes) {
 // allNodes[nodeId].color = "purple";
 allNodes[nodeId].color = nodeColors[nodeId];
 // delete allNodes[nodeId].color;
 if (allNodes[nodeId].hiddenLabel !== undefined) {
 allNodes[nodeId].label = allNodes[nodeId].hiddenLabel;
 allNodes[nodeId].hiddenLabel = undefined;
 }
 }
 highlightActive = false;
 }

 // transform the object into an array
 var updateArray = [];
 if (params.nodes.length > 0) {
 for (let nodeId in allNodes) {
 if (allNodes.hasOwnProperty(nodeId)) {
 // console.log(allNodes[nodeId]);
 updateArray.push(allNodes[nodeId]);
 }
 }
 nodes.update(updateArray);
 } else {
 // console.log("Nothing was selected");
 for (let nodeId in allNodes) {
 if (allNodes.hasOwnProperty(nodeId)) {
 // console.log(allNodes[nodeId]);
 // allNodes[nodeId].color = {};
 updateArray.push(allNodes[nodeId]);
 }
 }
 nodes.update(updateArray);
 }
}

function filterHighlight(params) {
 allNodes = nodes.get({ returnType: "Object" });
 // if something is selected:
 if (params.nodes.length > 0) {
 filterActive = true;
 let selectedNodes = params.nodes;

 // hiding all nodes and saving the label
 for (let nodeId in allNodes) {
 allNodes[nodeId].hidden = true;
 if (allNodes[nodeId].savedLabel === undefined) {
 allNodes[nodeId].savedLabel = allNodes[nodeId].label;
 allNodes[nodeId].label = undefined;
 }
 }

 for (let i=0; i < selectedNodes.length; i++) {
 allNodes[selectedNodes[i]].hidden = false;
 if (allNodes[selectedNodes[i]].savedLabel !== undefined) {
 allNodes[selectedNodes[i]].label = allNod